In [1]:
import pandas as pd

products = pd.read_parquet("../data/processed/products_clean.parquet")
behavior = pd.read_parquet("../data/processed/user_behavior_clean.parquet")

In [2]:
purchases = behavior[behavior['event_type'] == 'purchase'].copy()

In [8]:
purchases = purchases.merge(
    products[['product_id', 'price']],
    on='product_id',
    how='left'
)
purchases['timestamp'] = pd.to_datetime(purchases['timestamp'])

In [9]:
reference_date = pd.to_datetime('2023-12-31')

In [10]:
recency = purchases.groupby('user_id')['timestamp'].max().reset_index()
recency['recency'] = (reference_date - recency['timestamp']).dt.days

recency = recency[['user_id', 'recency']]

In [11]:
frequency = purchases.groupby('user_id').size().reset_index(name='frequency')

In [14]:
monetary = purchases.groupby('user_id')['price_x'].sum().reset_index()
monetary.rename(columns={'price_x': 'monetary'}, inplace=True)

In [16]:
rfm = recency.merge(frequency, on='user_id')
rfm = rfm.merge(monetary, on='user_id')

In [17]:
all_users = behavior[['user_id']].drop_duplicates()

rfm = all_users.merge(rfm, on='user_id', how='left')


In [18]:
rfm.fillna({
    'recency': 999,
    'frequency': 0,
    'monetary': 0
}, inplace=True)



In [19]:
print(rfm.head())
print(rfm.describe())



   user_id  recency  frequency  monetary
0     1861    999.0        0.0       0.0
1     1276   -594.0        3.0    2254.0
2      645   -482.0        1.0     460.0
3     1184    999.0        0.0       0.0
4     1342    999.0        0.0       0.0
           user_id      recency    frequency     monetary
count  2000.000000  2000.000000  2000.000000  2000.000000
mean   1000.500000   151.924500     0.709500   422.428000
std     577.494589   832.184923     0.833341   577.265546
min       1.000000  -821.000000     0.000000     0.000000
25%     500.750000  -673.000000     0.000000     0.000000
50%    1000.500000  -468.500000     1.000000    79.000000
75%    1500.250000   999.000000     1.000000   752.250000
max    2000.000000   999.000000     6.000000  3913.000000


In [20]:
rfm.to_parquet("../data/processed/user_features.parquet", index=False)